In [ ]:
dbutils.widgets.text("catalog_param", "my_assessment")
dbutils.widgets.text("schema_param", "silver")

catalog = dbutils.widgets.get("catalog_param")
schema = dbutils.widgets.get("schema_param")

In [ ]:
from pyspark.sql.functions import col, current_timestamp, lit, max
from delta.tables import DeltaTable
from datetime import datetime

In [ ]:
# We store the last run time in a Delta table
# Every run reads this, processes only new/changed records
# Then updates it with current time

watermark_table = f"{catalog}.silver.pipeline_watermark"

if not spark.catalog.tableExists(watermark_table):
    spark.sql(f"""
        CREATE TABLE {watermark_table} (
            table_name STRING,
            last_run_time TIMESTAMP
        )
    """)
    
    spark.sql(f"""
        INSERT INTO {watermark_table} VALUES
        ('customers',   '1900-01-01 00:00:00'),
        ('products',    '1900-01-01 00:00:00'),
        ('orders',      '1900-01-01 00:00:00'),
        ('order_items', '1900-01-01 00:00:00')
    """)
    print("Watermark table created")
else:
    print("Watermark table exists")

In [ ]:
def get_last_run_time(table_name):
    result = spark.sql(f"""
        SELECT last_run_time 
        FROM {watermark_table}
        WHERE table_name = '{table_name}'
    """).collect()
    return result[0][0]

In [ ]:
def update_watermark(table_name, run_time):
    spark.sql(f"""
        UPDATE {watermark_table}
        SET last_run_time = '{run_time}'
        WHERE table_name = '{table_name}'
    """)
    print(f"Watermark updated for {table_name} to {run_time}")

In [ ]:
def incremental_merge(table_name, primary_key):
    
    print(f"\n Processing {table_name}")
    
    last_run_time = get_last_run_time(table_name)
    current_run_time = datetime.now()
    print(f"   Last run time : {last_run_time}")
    print(f"   Current time  : {current_run_time}")

    df_bronze = spark.table(f"{catalog}.bronze.{table_name}")

    df_incremental = df_bronze.filter(
        (col("created_at") > lit(last_run_time)) |
        (col("updated_at") > lit(last_run_time))
    )
    
    incremental_count = df_incremental.count()
    print(f"   New/changed records found: {incremental_count}")
    
    if incremental_count == 0:
        print(f" No new records — skipping {table_name}")
        return
    
    df_cleaned = df_incremental \
        .dropDuplicates([primary_key]) \
        .dropna(subset=[primary_key]) \
        .drop("ingestion_date", "source_path")
    
    full_table_name = f"{catalog}.silver.{table_name}_cleaned"
    
    if spark.catalog.tableExists(full_table_name):
        delta_table = DeltaTable.forName(spark, full_table_name)
        
        delta_table.alias("target").merge(
            df_cleaned.alias("source"),
            f"target.{primary_key} = source.{primary_key}"
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
        print(f"MERGED {incremental_count} records into {full_table_name}")
    
    else:
        # Table doesn't exist → full load
        df_cleaned.write.mode("overwrite") \
            .saveAsTable(full_table_name)
        print(f"CREATED {full_table_name} with {incremental_count} records")
    
    update_watermark(table_name, current_run_time)

In [ ]:
incremental_merge("customers",primary_key="customer_id")
incremental_merge("products",primary_key="product_id")
incremental_merge("orders",primary_key="order_id")
incremental_merge("order_items",primary_key="order_item_id")